# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are healthy as part of a balanced diet.
    
    Key nutrients per medium banana (about 118 g):
    - Potassium: supports blood pressure and heart function
    - Vitamin B6: helps metabolism and brain function
    - Vitamin C: antioxidant and immune support
    - Fiber: aids digestion (about 3 g)
    
    Considerations:
    - Natural sugars/carb load (~24 g) and about 105 calories—watch portion if you’re managing blood sugar or calories.
    - Pair with protein/fat to improve satiety and sugar control.
    - Most people can eat 1–2 per day; adjust for your total intake and goals.
    
    Storage: keep at room temp until ripe; refrigerate to slow further ripening.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Overall, yes—bananas are healthy in moderation.

Key positives:
- Good source of potassium (about 400–450 mg per medium fruit) and vitamin C.
- 3 g fiber, which supports digestion.
- Provides quick energy from natural sugars and starch.

Considerations:
- Moderate glycemic impact; ripe bananas have more sugar.
- Not a complete protein; vary fruit intake with other foods.
- if you have diabetes, watch portions as part of total carb intake.

Practical: 1–2 bananas daily fit most balanced diets. Pair with protein or fat for staying power.

_Good Job!_